In [38]:
import json

In [39]:
with open('/home/simonettos/thijs/classification/classification/datasets/enterprise-attack.json', 'r') as f:
    data = json.load(f)

In [50]:
import json
from pathlib import Path

def extract_attack_techniques(bundle_path, include_revoked=False, include_deprecated=False, domains=None):
    """
    domains: None or a set/list like {"enterprise-attack"} or {"ics-attack"} or {"mobile-attack"}
    Returns: dict keyed by external_id (e.g., T1059, T1059.003)
    """
    bundle = json.loads(Path(bundle_path).read_text(encoding="utf-8"))
    objs = bundle.get("objects", [])

    out = {}
    for o in objs:
        if o.get("type") != "attack-pattern":
            continue

        # Filters
        if not include_revoked and o.get("revoked", False):
            continue
        if not include_deprecated and o.get("x_mitre_deprecated", False):
            continue

        # Optional domain filter (Enterprise / Mobile / ICS)
        if domains is not None:
            obj_domains = set(o.get("x_mitre_domains", []) or [])
            if obj_domains.isdisjoint(set(domains)):
                continue

        # Find the ATT&CK external_id (Txxxx or Txxxx.xxx)
        ext_id = None
        ext_url = None
        for ref in o.get("external_references", []) or []:
            if ref.get("source_name") == "mitre-attack" and "external_id" in ref:
                ext_id = ref["external_id"]
                ext_url = ref.get("url")
                break

        if not ext_id:
            continue  # not a technique/sub-technique in ATT&CK terms

        out[ext_id] = {
            "stix_id": o.get("id"),
            "name": o.get("name"),
            "description": o.get("description", ""),
            "url": ext_url,
            "is_subtechnique": bool(o.get("x_mitre_is_subtechnique", False)),
            "domains": o.get("x_mitre_domains", []),
            "revoked": bool(o.get("revoked", False)),
            "deprecated": bool(o.get("x_mitre_deprecated", False)),
        }

    return out


if __name__ == "__main__":
    bundle_path = "/home/simonettos/thijs/classification/classification/datasets/enterprise-attack.json"  # <- change to your downloaded bundle file
    techniques = extract_attack_techniques(
        bundle_path,
        include_revoked=False,
        include_deprecated=False,
        domains={"enterprise-attack"}  # or {"ics-attack"} / {"mobile-attack"} / None for all
    )

    print(f"Extracted {len(techniques)} techniques/sub-techniques.")
    # show a couple
    for k in sorted(list(techniques.keys()))[:5]:
        print(k, "-", techniques[k]["name"])

    # Save to JSON
    Path("techniques.json").write_text(json.dumps(techniques, indent=2, ensure_ascii=False), encoding="utf-8")

    # Save to CSV (id, name, description)
    import csv
    with open("techniques.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["technique_id", "name", "description", "url", "is_subtechnique", "domains"])
        for tid in sorted(techniques.keys()):
            t = techniques[tid]
            w.writerow([tid, t["name"], t["description"], t["url"], t["is_subtechnique"], ";".join(t["domains"])])


Extracted 637 techniques/sub-techniques.
T1001 - Data Obfuscation
T1001.001 - Junk Data
T1001.002 - Steganography
T1001.003 - Protocol Impersonation
T1003 - OS Credential Dumping


In [51]:
print(len(techniques))

637
